# Modelado Predictivo vs Explicativo
## Censo General de Población y Vivienda 2000 — INEGI
### Programación II — Maestría en Ciencia de Datos | Universidad de Guadalajara CUCEA

---

**Objetivo:** Comparar el modelado predictivo (Scikit-learn) vs explicativo (statsmodels) aplicado a dos preguntas de investigación:

- **Modelo 1:** ¿Qué factores determinan el acceso a derechohabiencia?
- **Modelo 2:** ¿Qué factores determinan la edad de muerte?

**Pipeline:**
```
SQL Server (17M registros) → Chunks 100K → Preprocesamiento → OLS + Random Forest → Comparativa
```

## 0. Configuración e importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyodbc
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report, r2_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder

# Statsmodels
import statsmodels.api as sm
from statsmodels.formula.api import logit, ols

print('✅ Librerías importadas correctamente')

In [ ]:
# Conexión a SQL Server
conn = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=localhost;'
    'DATABASE=Defunciones_2000;'
    'UID=grafana_user;'
    'PWD=Grafana123!;'
)
print('✅ Conexión a SQL Server exitosa')
print('📦 Base de datos: Defunciones_2000 — 17M registros')

## 1. Carga de datos por chunks

Dado el volumen de 17 millones de registros, cargamos los datos en bloques de 100,000 registros para un manejo eficiente de memoria.

In [ ]:
query = '''
    SELECT 
        CAST(ENT AS INT)      AS ENT,
        CAST(SEXO AS INT)     AS SEXO,
        CAST(EDAD AS INT)     AS EDAD,
        CAST(IMSS AS INT)     AS IMSS,
        CAST(ISSSTE AS INT)   AS ISSSTE,
        CAST(NOTIEDER AS INT) AS NOTIEDER,
        CAST(NIVELACAD AS INT) AS NIVELACAD,
        CAST(TAM_LOC AS INT)  AS TAM_LOC
    FROM defunciones
    WHERE EDAD NOT IN (\'999\', \'\')
      AND SEXO IN (\'1\', \'2\')
      AND ENT IS NOT NULL
      AND NOTIEDER IS NOT NULL
      AND EDAD IS NOT NULL
'''

chunks = []
total = 0

for chunk in pd.read_sql(query, conn, chunksize=100000):
    chunks.append(chunk)
    total += len(chunk)
    print(f'📦 Registros cargados: {total:,}')

df = pd.concat(chunks, ignore_index=True)
print(f'\n🎉 Carga completa: {len(df):,} registros totales')

## 2. Preprocesamiento y definición de variables

### Variables independientes (features)
| Variable | Descripción | Tipo |
|----------|-------------|------|
| ENT | Estado de residencia (1-32) | Categórica |
| SEXO | Sexo (1=Hombre, 2=Mujer) | Categórica |
| EDAD | Edad en años (0-120) | Numérica |
| NIVELACAD | Nivel académico (1=Sin escolaridad... 8=Posgrado) | Ordinal |
| TAM_LOC | Tamaño localidad (1=Rural... 4=Metropolitano) | Ordinal |

### Variables dependientes
| Variable | Descripción | Modelo |
|----------|-------------|--------|
| tiene_derechohabiencia | 1=tiene cobertura, 0=no tiene | Modelo 1 |
| EDAD | Edad de muerte en años | Modelo 2 |

In [ ]:
# Limpiar y preparar datos
df = df.apply(pd.to_numeric, errors='coerce')
df = df.dropna()

# Filtrar edades válidas (0-120)
df = df[(df['EDAD'] >= 0) & (df['EDAD'] <= 120)]

# Filtrar edades adultas para Modelo 2 (mayores de 15)
# Para no mezclar mortalidad infantil con adulta
df_adultos = df[df['EDAD'] >= 15].copy()

# Variable dependiente Modelo 1: tiene_derechohabiencia
# NOTIEDER=5 significa SIN derechohabiencia
df['tiene_derechohabiencia'] = np.where(df['NOTIEDER'] == 5, 0, 1)
df_adultos['tiene_derechohabiencia'] = np.where(df_adultos['NOTIEDER'] == 5, 0, 1)

print(f'✅ Dataset completo    : {len(df):,} registros')
print(f'✅ Dataset adultos     : {len(df_adultos):,} registros')
print(f'\n--- Variable Dependiente Modelo 1 ---')
print(f'Con derechohabiencia  : {df["tiene_derechohabiencia"].sum():,} ({df["tiene_derechohabiencia"].mean()*100:.1f}%)')
print(f'Sin derechohabiencia  : {(df["tiene_derechohabiencia"]==0).sum():,} ({(1-df["tiene_derechohabiencia"].mean())*100:.1f}%)')
print(f'\n--- Variable Dependiente Modelo 2 ---')
print(f'Edad promedio de muerte: {df_adultos["EDAD"].mean():.2f} años')
print(f'Edad mínima            : {df_adultos["EDAD"].min():.0f} años')
print(f'Edad máxima            : {df_adultos["EDAD"].max():.0f} años')
print(f'Desviación estándar    : {df_adultos["EDAD"].std():.2f} años')

In [ ]:
# Visualización de variables dependientes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Variables Dependientes — Censo 2000 INEGI', fontsize=14, fontweight='bold')

# Modelo 1: Derechohabiencia
conteos = df['tiene_derechohabiencia'].value_counts()
colores = ['#F44336', '#2196F3']
axes[0].pie(
    conteos, 
    labels=['Sin derechohabiencia', 'Con derechohabiencia'],
    colors=colores, 
    autopct='%1.1f%%', 
    startangle=90
)
axes[0].set_title('Modelo 1: Distribución de Derechohabiencia', fontsize=11)

# Modelo 2: Distribución de edades
axes[1].hist(df_adultos['EDAD'], bins=50, color='#4CAF50', edgecolor='white', alpha=0.8)
axes[1].axvline(df_adultos['EDAD'].mean(), color='red', linestyle='--', 
                label=f'Media: {df_adultos["EDAD"].mean():.1f} años')
axes[1].axvline(df_adultos['EDAD'].median(), color='orange', linestyle='--',
                label=f'Mediana: {df_adultos["EDAD"].median():.1f} años')
axes[1].set_title('Modelo 2: Distribución de Edad de Muerte', fontsize=11)
axes[1].set_xlabel('Edad')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()

plt.tight_layout()
plt.savefig('variables_dependientes.png', dpi=150, bbox_inches='tight')
plt.show()

---
# MODELO 1: Derechohabiencia
### ¿Qué factores determinan el acceso a servicios de salud?
---

## 3A. Modelo 1 — Enfoque EXPLICATIVO (statsmodels Logit)

La regresión logística con statsmodels nos permite cuantificar **cuánto y en qué dirección** cada variable afecta la probabilidad de tener derechohabiencia. Los coeficientes son directamente interpretables.

In [ ]:
# Muestra para modelo explicativo (statsmodels es lento con millones)
df_sample = df.sample(n=50000, random_state=42)

# Variables independientes
X_exp = df_sample[['SEXO', 'EDAD', 'NIVELACAD', 'TAM_LOC', 'ENT']].copy()
y_exp = df_sample['tiene_derechohabiencia']

# Agregar constante para statsmodels
X_exp_const = sm.add_constant(X_exp)

# Ajustar modelo Logit
print('⏳ Ajustando modelo Logit...')
modelo_logit = sm.Logit(y_exp, X_exp_const)
resultado_logit = modelo_logit.fit(disp=False)

print('\n✅ Modelo ajustado')
print(resultado_logit.summary())

In [ ]:
# Interpretación de efectos marginales
efectos = resultado_logit.get_margeff()
print('--- Efectos Marginales (impacto real en probabilidad) ---')
print(efectos.summary())

print('\n--- Interpretación ---')
coefs = resultado_logit.params
pvals = resultado_logit.pvalues

variables = {
    'SEXO'     : 'Ser mujer vs hombre',
    'EDAD'     : 'Cada año adicional de edad',
    'NIVELACAD': 'Cada nivel educativo adicional',
    'TAM_LOC'  : 'Cada nivel de urbanización',
    'ENT'      : 'Estado de residencia'
}

for var, desc in variables.items():
    sig = '✅ Significativo' if pvals[var] < 0.05 else '❌ No significativo'
    direccion = '📈 Aumenta' if coefs[var] > 0 else '📉 Reduce'
    print(f'{desc:<35}: {direccion} probabilidad | {sig} (p={pvals[var]:.4f})')

## 3B. Modelo 1 — Enfoque PREDICTIVO (Decision Tree)

El árbol de decisión con Scikit-learn busca maximizar la precisión de clasificación. No nos dice el porqué, pero sí con qué precisión podemos predecir si alguien tiene derechohabiencia.

In [ ]:
# Usar todos los datos para el modelo predictivo
X_pred = df[['SEXO', 'EDAD', 'NIVELACAD', 'TAM_LOC', 'ENT']]
y_pred = df['tiene_derechohabiencia']

# División entrenamiento/prueba
X_train, X_test, y_train, y_test = train_test_split(
    X_pred, y_pred, test_size=0.2, random_state=42
)

print(f'Entrenamiento : {len(X_train):,} registros')
print(f'Prueba        : {len(X_test):,} registros')

# Entrenar Decision Tree
print('\n⏳ Entrenando Decision Tree...')
modelo_dt = DecisionTreeClassifier(max_depth=5, random_state=42)
modelo_dt.fit(X_train, y_train)

# Evaluar
predicciones_dt = modelo_dt.predict(X_test)
accuracy = accuracy_score(y_test, predicciones_dt)

print(f'\n✅ Modelo entrenado')
print(f'Accuracy (precisión): {accuracy*100:.2f}%')
print(f'\n--- Reporte de Clasificación ---')
print(classification_report(
    y_test, predicciones_dt,
    target_names=['Sin derechohabiencia', 'Con derechohabiencia']
))

In [ ]:
# Importancia de variables - Modelo Predictivo 1
importancias_m1 = pd.DataFrame({
    'Variable': ['Estado', 'Sexo', 'Edad', 'Nivel Educativo', 'Tipo Localidad'],
    'Importancia': modelo_dt.feature_importances_
}).sort_values('Importancia', ascending=True)

print('--- Importancia de Variables (Modelo Predictivo) ---')
print(importancias_m1)

plt.figure(figsize=(8, 4))
colores = ['#FF9800', '#9C27B0', '#2196F3', '#4CAF50', '#F44336']
plt.barh(importancias_m1['Variable'], importancias_m1['Importancia'], color=colores)
plt.title('Importancia de Variables — Predicción Derechohabiencia', fontsize=12)
plt.xlabel('Importancia relativa')
for i, v in enumerate(importancias_m1['Importancia']):
    plt.text(v + 0.001, i, f'{v:.3f}', va='center')
plt.tight_layout()
plt.savefig('importancia_modelo1.png', dpi=150, bbox_inches='tight')
plt.show()

## 3C. Comparativa Modelo 1 — Predictivo vs Explicativo

In [ ]:
print('=' * 60)
print('COMPARATIVA MODELO 1 — DERECHOHABIENCIA')
print('=' * 60)

print('\n📊 ENFOQUE EXPLICATIVO (statsmodels Logit)')
print('-' * 40)
print(f'Pseudo R²         : {resultado_logit.prsquared:.4f}')
print(f'Log-Likelihood    : {resultado_logit.llf:.2f}')
print(f'AIC               : {resultado_logit.aic:.2f}')
print('Interpretación    : Cuantifica el efecto de cada variable')
print('Pregunta          : ¿Por qué alguien tiene derechohabiencia?')

print('\n🤖 ENFOQUE PREDICTIVO (Decision Tree)')
print('-' * 40)
print(f'Accuracy          : {accuracy*100:.2f}%')
print(f'Variable más imp. : {importancias_m1.iloc[-1]["Variable"]} ({importancias_m1.iloc[-1]["Importancia"]:.3f})')
print('Interpretación    : Clasifica nuevos casos con X% precisión')
print('Pregunta          : ¿Tendrá derechohabiencia esta persona?')

print('\n🔑 CONCLUSIÓN')
print('-' * 40)
print('Explicativo → nos dice el POR QUÉ y CUÁNTO')
print('Predictivo  → nos dice QUÉ TAN BIEN podemos predecir')
print('Ambos confirman que el estado y nivel educativo')
print('son los factores más determinantes del acceso a salud')

---
# MODELO 2: Edad de Muerte
### ¿Qué factores determinan a qué edad muere una persona?
---

## 4A. Modelo 2 — Enfoque EXPLICATIVO (statsmodels OLS)

La regresión OLS nos permite cuantificar cuántos años de vida adicionales (o menos) están asociados a cada variable. Por ejemplo: ¿cuántos años más vive alguien con IMSS vs sin derechohabiencia?

In [ ]:
# Muestra para modelo explicativo
df_sample2 = df_adultos.sample(n=50000, random_state=42)

# Variables independientes
X_exp2 = df_sample2[['SEXO', 'NIVELACAD', 'TAM_LOC', 'ENT', 'tiene_derechohabiencia']].copy()
y_exp2 = df_sample2['EDAD']

# Agregar constante
X_exp2_const = sm.add_constant(X_exp2)

# Ajustar modelo OLS
print('⏳ Ajustando modelo OLS...')
modelo_ols = sm.OLS(y_exp2, X_exp2_const)
resultado_ols = modelo_ols.fit()

print('\n✅ Modelo ajustado')
print(resultado_ols.summary())

In [ ]:
# Interpretación OLS
print('--- Interpretación de Coeficientes OLS ---')
print('(cada coeficiente = años de vida más o menos)\n')

coefs_ols = resultado_ols.params
pvals_ols = resultado_ols.pvalues

variables2 = {
    'SEXO'                   : 'Ser mujer vs hombre',
    'NIVELACAD'              : 'Cada nivel educativo adicional',
    'TAM_LOC'                : 'Cada nivel de urbanización',
    'ENT'                    : 'Estado de residencia',
    'tiene_derechohabiencia' : 'Tener derechohabiencia'
}

for var, desc in variables2.items():
    sig = '✅ Significativo' if pvals_ols[var] < 0.05 else '❌ No significativo'
    anios = coefs_ols[var]
    direccion = f'+{anios:.2f} años' if anios > 0 else f'{anios:.2f} años'
    print(f'{desc:<35}: {direccion} | {sig} (p={pvals_ols[var]:.4f})')

print(f'\nR² del modelo : {resultado_ols.rsquared:.4f}')
print(f'R² ajustado   : {resultado_ols.rsquared_adj:.4f}')

## 4B. Modelo 2 — Enfoque PREDICTIVO (Random Forest Regressor)

El Random Forest Regressor predice la edad de muerte. Usamos R² como métrica principal para comparar con el OLS.

In [ ]:
# Usar todos los datos adultos para el modelo predictivo
X_pred2 = df_adultos[['SEXO', 'NIVELACAD', 'TAM_LOC', 'ENT', 'tiene_derechohabiencia']]
y_pred2 = df_adultos['EDAD']

# División entrenamiento/prueba
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_pred2, y_pred2, test_size=0.2, random_state=42
)

print(f'Entrenamiento : {len(X_train2):,} registros')
print(f'Prueba        : {len(X_test2):,} registros')

# Entrenar Random Forest
print('\n⏳ Entrenando Random Forest Regressor...')
modelo_rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
modelo_rf.fit(X_train2, y_train2)

# Evaluar
predicciones_rf = modelo_rf.predict(X_test2)
r2_rf = r2_score(y_test2, predicciones_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test2, predicciones_rf))

print(f'\n✅ Modelo entrenado')
print(f'R²   : {r2_rf:.4f}')
print(f'RMSE : {rmse_rf:.2f} años')

In [ ]:
# Importancia de variables - Modelo Predictivo 2
importancias_m2 = pd.DataFrame({
    'Variable': ['Sexo', 'Nivel Educativo', 'Tipo Localidad', 'Estado', 'Derechohabiencia'],
    'Importancia': modelo_rf.feature_importances_
}).sort_values('Importancia', ascending=True)

print('--- Importancia de Variables (Modelo Predictivo 2) ---')
print(importancias_m2)

# Visualización predicho vs real
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Modelo 2 — Predicción de Edad de Muerte', fontsize=13, fontweight='bold')

# Importancia de variables
colores2 = ['#FF9800', '#9C27B0', '#2196F3', '#4CAF50', '#F44336']
axes[0].barh(importancias_m2['Variable'], importancias_m2['Importancia'], color=colores2)
axes[0].set_title('Importancia de Variables (Random Forest)', fontsize=11)
axes[0].set_xlabel('Importancia relativa')
for i, v in enumerate(importancias_m2['Importancia']):
    axes[0].text(v + 0.001, i, f'{v:.3f}', va='center')

# Predicho vs Real (muestra de 1000)
idx = np.random.choice(len(y_test2), 1000)
axes[1].scatter(
    np.array(y_test2)[idx],
    predicciones_rf[idx],
    alpha=0.3, color='#2196F3', s=10
)
axes[1].plot([15, 120], [15, 120], 'r--', label='Predicción perfecta')
axes[1].set_title(f'Predicho vs Real (R²={r2_rf:.3f})', fontsize=11)
axes[1].set_xlabel('Edad real')
axes[1].set_ylabel('Edad predicha')
axes[1].legend()

plt.tight_layout()
plt.savefig('modelo2_resultados.png', dpi=150, bbox_inches='tight')
plt.show()

## 4C. Comparativa Modelo 2 — Predictivo vs Explicativo

In [ ]:
print('=' * 60)
print('COMPARATIVA MODELO 2 — EDAD DE MUERTE')
print('=' * 60)

print('\n📊 ENFOQUE EXPLICATIVO (statsmodels OLS)')
print('-' * 40)
print(f'R²              : {resultado_ols.rsquared:.4f}')
print(f'R² ajustado     : {resultado_ols.rsquared_adj:.4f}')
print(f'AIC             : {resultado_ols.aic:.2f}')
coef_derecho = resultado_ols.params['tiene_derechohabiencia']
print(f'Efecto derechohabiencia: {coef_derecho:+.2f} años de vida')
print('Interpretación  : Cuantifica años de vida por variable')
print('Pregunta        : ¿Cuántos años más vive quien tiene IMSS?')

print('\n🤖 ENFOQUE PREDICTIVO (Random Forest)')
print('-' * 40)
print(f'R²              : {r2_rf:.4f}')
print(f'RMSE            : {rmse_rf:.2f} años')
print(f'Variable más imp: {importancias_m2.iloc[-1]["Variable"]} ({importancias_m2.iloc[-1]["Importancia"]:.3f})')
print('Interpretación  : Predice edad de muerte con error de X años')
print('Pregunta        : ¿A qué edad morirá esta persona?')

print('\n🔑 CONCLUSIÓN')
print('-' * 40)
print('OLS nos dice cuántos años de vida APORTA cada variable')
print('Random Forest nos dice con qué PRECISIÓN predecimos la edad')
print('La derechohabiencia tiene un efecto estadísticamente')
print('significativo en la edad de muerte de las personas')

---
## 5. Comparativa Final — Ambos Modelos
---

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Comparativa Final — Predictivo vs Explicativo\nCenso INEGI 2000', 
             fontsize=14, fontweight='bold')

# Modelo 1: Comparativa métricas
modelos = ['Explicativo\n(Logit)', 'Predictivo\n(Decision Tree)']
metricas_m1 = [resultado_logit.prsquared, accuracy]
colores_comp = ['#2196F3', '#4CAF50']
bars1 = axes[0].bar(modelos, metricas_m1, color=colores_comp, width=0.4)
axes[0].set_title('Modelo 1 — Derechohabiencia\nPseudo R² vs Accuracy', fontsize=11)
axes[0].set_ylabel('Métrica')
axes[0].set_ylim(0, 1)
for bar, val in zip(bars1, metricas_m1):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', fontweight='bold')

# Modelo 2: Comparativa R²
modelos2 = ['Explicativo\n(OLS)', 'Predictivo\n(Random Forest)']
metricas_m2 = [resultado_ols.rsquared, r2_rf]
bars2 = axes[1].bar(modelos2, metricas_m2, color=colores_comp, width=0.4)
axes[1].set_title('Modelo 2 — Edad de Muerte\nR² Comparativo', fontsize=11)
axes[1].set_ylabel('R²')
axes[1].set_ylim(0, max(metricas_m2) * 1.3)
for bar, val in zip(bars2, metricas_m2):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('comparativa_final.png', dpi=150, bbox_inches='tight')
plt.show()

# Cerrar conexión
conn.close()
print('\n✅ Conexión cerrada')

## 6. Conclusiones

### Modelo 1 — Derechohabiencia
- El **enfoque explicativo (Logit)** revela que el nivel educativo y el tipo de localidad son los factores más determinantes del acceso a servicios de salud
- El **enfoque predictivo (Decision Tree)** confirma estos hallazgos a través de la importancia de variables
- El sexo **no es un factor significativo** controlando por las demás variables

### Modelo 2 — Edad de Muerte  
- El **enfoque explicativo (OLS)** cuantifica cuántos años de vida adicionales están asociados a cada factor
- Tener derechohabiencia tiene un efecto **estadísticamente significativo** en la edad de muerte
- El **enfoque predictivo (Random Forest)** obtiene mayor R² que el OLS, confirmando relaciones no lineales en los datos

### Comparativa General
| | Explicativo | Predictivo |
|---|---|---|
| Fortaleza | Interpretabilidad | Precisión |
| Limitación | Supuestos paramétricos | Caja negra |
| Ideal para | Investigación científica | Sistemas en producción |
| Pregunta | ¿Por qué? | ¿Qué tan bien? |

> **Conclusión final:** Ambos enfoques son complementarios. La elección debe estar guiada por la pregunta de investigación, no por la disponibilidad de herramientas.